In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, regularizers
from keras.optimizers import Adam

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
IMG_SIZE = (28, 28)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def get_dataset(image_dir):
    dataset = tf.keras.preprocessing.image_dataset_from_directory(
        image_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        color_mode='grayscale',  # Ensure images are loaded as grayscale
        label_mode='categorical'
    )
    return dataset

def augment_data(images, labels):
    images = tf.reshape(images, (-1, 28, 28, 1)) 
    augmented_images = data_augmentation(images, training=True)
    return augmented_images, labels

In [ ]:
dataset = get_dataset('/kaggle/input/mnist-synthetic-100000/all_labeled_images')

dataset_size = tf.data.experimental.cardinality(dataset).numpy()

In [ ]:
train_size = int(0.85 * dataset_size)  
val_size = int(0.15 * dataset_size)  

train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size).take(val_size)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2)
])

In [ ]:
train_ds = train_ds.map(augment_data, num_parallel_calls=AUTOTUNE)

In [ ]:
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])


In [ ]:
model.summary()

In [ ]:
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
model.fit(train_ds, validation_data=val_ds, verbose=1, epochs=50)

In [ ]:
dataset = get_dataset('/kaggle/input/mnist-test-data/trainingSet')

test_size = tf.data.experimental.cardinality(dataset).numpy()

test_ds = dataset.take(test_size)

AUTOTUNE = tf.data.AUTOTUNE

test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
model.evaluate(test_ds)